# Problem Statement

## Context

In the realm of modern finance, businesses encounter the perpetual challenge of managing debt obligations effectively to maintain a favorable credit standing and foster sustainable growth. Investors keenly scrutinize companies capable of navigating financial complexities while ensuring stability and profitability. A pivotal instrument in this evaluation process is the balance sheet, which provides a comprehensive overview of a company's assets, liabilities, and shareholder equity, offering insights into its financial health and operational efficiency. In this context, leveraging available financial data, particularly from preceding fiscal periods, becomes imperative for informed decision-making and strategic planning.

## Objective

A renowned credit rating organization wants to develop a Financial Health Assessment Tool. With the help of the tool, it endeavors to empower businesses and investors with a robust mechanism for evaluating the financial well-being and creditworthiness of companies. By harnessing machine learning techniques, the organization aims to analyze historical financial statements and extract pertinent insights to facilitate informed decision-making via the tool. Specifically, the organization foresees facilitating the following with the help of the tool:

1. Debt Management Analysis: Identify patterns and trends in debt management practices to assess the ability of businesses to fulfill financial obligations promptly and efficiently, and identify potential cases of default.

2. Credit Risk Evaluation: Evaluate credit risk exposure by analyzing liquidity ratios, debt-to-equity ratios, and other key financial indicators to ascertain the likelihood of default and inform investment decisions.

As a part of the data science team in the organization, you have been provided with the financial metrics of different companies. The task is to analyze the data provided and develop a predictive model leveraging machine learning techniques to identify whether a given company will default on its debt repayments in the next two quarters. The predictive model will help the organization anticipate potential challenges with the financial performance of the companies and enable proactive risk mitigation strategies.

## Data Dictionary

The data consists of financial metrics from the balance sheets of different companies. The detailed data dictionary is available in the data dictionary file (*FRA_DataDictionary.xlsx*).

# **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_______' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_______' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Fill the code wherever asked by the commented lines like "# write your code here" or "# complete the code". Running incomplete code may throw error.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same.

# Importing necessary libraries

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split, GridSearchCV    # Train test Split and Grid Search
from sklearn.ensemble import RandomForestClassifier

import statsmodels.api as SM
from sklearn import metrics

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

from IPython.core.display import display, HTML
display(HTML('<style>.container { width:90% !important; }<\style>'))

# Loading the Data

In [ ]:
# uncomment and run the following lines for Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
df = pd.read_excel('CompData.xlsx')  ##  Fill the blank to read the data
df.head()

# Data Overview

In [ ]:
df.head(5) ##  Complete the code to view top 5 rows of the data

In [ ]:
df.tail(5) ##  Complete the code to view last 5 rows of the data

In [ ]:
df.shape ##  Complete the code to view dimensions of the data

In [ ]:
df.info()

In [ ]:
# Remove '_' (startswith) from column headers where present
for col in df.columns:
    if col.startswith('_'):
        df.rename(columns={col: col[1:]}, inplace=True)

In [ ]:
df.info()

In [ ]:
# checking for duplicate values
df.duplicated().sum()##  Complete the code to check duplicate entries in the data

In [ ]:
#checking for no. of unique values
df.nunique() ##  Complete the code to check unique entries in the data

In [ ]:
df.describe().T

* We can see that `Co_Code` and `Co_Name` are not relevant for this exercise
* So we will drop these variables

In [ ]:
df.drop(['Co_Code', 'Co_Name'], axis = 1, inplace = True)

## Exploratory Data Analysis

## Univariate Analysis

In [ ]:
df["Default"].nunique()   ## Complete the code to check unique values in the mentioned column

In [ ]:
#Plotting a countplot for the target variable
sns.countplot(x = "Default", data = df, hue = 'Default')   ## Complete the code to get a countplot of the mentionedd column.
plt.title('Count of Default')
plt.show()

In [ ]:
#Percentage of defaulters
(df.Default.sum()/len(df)) * 100

In [ ]:
#Get boxplots for all the numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(18, 30))

# Loop through each numerical column and plot a boxplot
for i, variable in enumerate(numeric_columns):
    plt.subplot(15, 4, i + 1)
    sns.boxplot(x=df[variable])
    plt.title(variable)

plt.tight_layout()
plt.show()

In [ ]:
#Get distplot for all the numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(18, 30))

# Loop through each numerical column and plot a histogram
for i, variable in enumerate(numeric_columns):
    if df[variable].dtype != 'O':  # Skip non-numeric columns (extra safety)
        plt.subplot(15, 4, i + 1)
        sns.histplot(data=df, x=variable, kde=True, bins=30, color='skyblue')
        plt.title(variable)

# Adjust layout after all plots are created
plt.tight_layout()
plt.show()

## Bivariate Analysis

In [ ]:
#Get boxplots for all the numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(18, 30))

for i, variable in enumerate(numeric_columns):
    plt.subplot(15, 4, i + 1)
    sns.boxplot(df['Default'])  ## Complete the code to get boxplot of all variables with Default column in the data
    plt.tight_layout()
    plt.title(variable)


In [ ]:
# Calculate the correlation matrix
corr_matrix = df.corr(numeric_only=True)  ## Complete the code to get the correlation matrix for the data

# Create a heatmap of the correlation matrix
plt.figure(figsize=(50, 50))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".1f", annot_kws={"size": 15})
plt.title('Heatmap of Correlation Matrix')
plt.show()

# Data Preprocessing

## Dropping columns with few unique values

In [ ]:
df.nunique()

We can drop the columns `Net_Income_Flag` and `Liability_Assets_Flag` as they have very few unique values.

In [ ]:
df.drop(['Net_Income_Flag', 'Liability_Assets_Flag'], axis = 1, inplace = True)  ## Complete the code to drop the mentioned columns from the dataset
df.nunique()

## Outliers Check

In [ ]:
outliers_count = {}

# Iterate over each column in the DataFrame
for column in df.columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR  ## Fill the blank with correct value for getting lower_bound
    upper_bound = Q3 + 1.5 * IQR  ## Fill the blank with correct value for getting upper_bound

    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    outliers_count[column] = len(outliers)

print("Number of outliers in each column:")
pd.DataFrame([{'Column': column, 'No. of outliers': outliers} for column, outliers in outliers_count.items()])

## Data Preparation for Modeling

In [ ]:
# Seperating target variable from the rest of the data
df_X = df.drop(['Default'], axis = 1)
df_y = df['Default']

In [ ]:
#Splitting the data for training and testing
X_train, X_test, y_train, y_test = train_test_split(df_X, df_y, test_size=0.25, random_state=42, stratify = df_y)  ## Complete the code to split the data into train and test in the ratio 75:25

## Missing Values Detection and Treatment

In [ ]:
# Check missing values
X_train.isnull().sum()  ## Complete the code to get the number of null or NaN values in each column

In [ ]:
# Check missing values
X_test.isnull().sum()

In [ ]:
#Replace the missing values in the data using KNN Imputer
KNNimputerModel = KNNImputer(n_neighbors=5)  ## Complete the code to select 5 neighbors for KNN Imputer

X_train = pd.DataFrame(KNNimputerModel.fit_transform(X_train), columns = X_train.columns)
X_test = pd.DataFrame(KNNimputerModel.transform(X_test), columns = X_test.columns)  ## Complete the code to replace missing values in X_test

In [ ]:
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

## Scaling the Data

In [ ]:
#Scaling of features is done to bring all the features to the same scale.
sc = StandardScaler()

X_train_scaled = pd.DataFrame(sc.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(sc.transform(X_test), columns=X_test.columns)  ## Complete the code to scale X_test to the same scale as X_train

In [ ]:
X_train_scaled.head()

In [ ]:
X_test_scaled.head()

## Model Building

## Model Evaluation Criterion

*Metric of Choice*
-


In [ ]:
# defining a function to compute different metrics to check performance of a classification model built using sklearn


def model_performance_classification(model, predictors, target, threshold = 0.5):
    """
    Function to compute different metrics to check classification model performance

    model: classifier
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    y_pred = model.predict(predictors)

    if len(list(set(y_pred))) != 2:
        y_prob_pred = model.predict(predictors)

        y_pred=[]
        for i in range(0,len(y_prob_pred)):
            if np.array(y_prob_pred)[i] > threshold:
                a=1
            else:
                a=0
            y_pred.append(a)
    else:
        pass

    acc = accuracy_score(target, y_pred)  # to compute Accuracy
    recall = recall_score(target, y_pred)  # to compute Recall
    precision = precision_score(target, y_pred)  # to compute Precision
    f1 = f1_score(target, y_pred)  # to compute F1-score

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {"Accuracy": acc, "Recall": recall, "Precision": precision, "F1": f1,},
        index=[0],
    )

    return df_perf

In [ ]:
def model_confusion_matrix(model, predictors, target, threshold = 0.5):
    """
    To plot the confusion_matrix with percentages

    model: classifier
    predictors: independent variables
    target: dependent variable
    """
    y_pred = model.predict(predictors)
    if len(list(set(y_pred))) != 2:
        y_prob_pred = model.predict(predictors)

        y_pred=[]
        for i in range(0,len(y_prob_pred)):
            if np.array(y_prob_pred)[i] > threshold:
                a=1
            else:
                a=0
            y_pred.append(a)
    else:
        pass

    cm = confusion_matrix(target, y_pred)
    labels = np.asarray(
        [
            ["{0:0.0f}".format(item) + "\n{0:.2%}".format(item / cm.flatten().sum())]
            for item in cm.flatten()
        ]
    ).reshape(2, 2)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=labels, fmt="")
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

## Logistic Regression

In [ ]:
# Adding constant to data for Logistic Regression
X_train_with_intercept = SM.add_constant(X_train_scaled)
X_test_with_intercept = SM.add_constant(X_test_scaled)

In [ ]:
X_train_with_intercept.head()

In [ ]:
y_train.reset_index(inplace = True, drop = True)

In [ ]:
LogisticReg = SM.Logit(y_train, X_train_with_intercept) ## Complete the code to define Logistic Regression Model

# Fit the model
LogisticReg_result = LogisticReg.fit()

print(LogisticReg_result.summary())

### Logistic Regression Model - Training Performance

In [ ]:
model_confusion_matrix(LogisticReg_result, X_train_with_intercept, y_train)

In [ ]:
logistic_regression_perf_train = model_performance_classification(LogisticReg_result, X_train_with_intercept, y_train)
logistic_regression_perf_train

### Logistic Regression Model - Test Performance

In [ ]:
model_confusion_matrix(LogisticReg_result,X_test_with_intercept,y_test)  ## Complete the code to create confusion matrix for test data

In [ ]:
logistic_regression_perf_test = model_performance_classification(LogisticReg_result,X_test_with_intercept,y_test)  ## Complete the code to check performance on test data
logistic_regression_perf_test

## Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [5, 7, 9],    # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],    # Minimum number of samples required to split a node
    'min_samples_leaf': [5, 6, 7],  # Minimum number of samples required at each leaf node
}
rf_classifier = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(estimator = rf_classifier, param_grid=param_grid)

In [ ]:
## Complete the code to define random forest with random state = 42
rf_model =rf_classifier.fit(X_train, y_train) ## Complete the code to fit random forest on the train data

### Random Forest Model - Training Performance

In [ ]:
model_confusion_matrix(rf_model, X_train, y_train)

In [ ]:
random_forest_perf_train = model_performance_classification(rf_model, X_train, y_train)
random_forest_perf_train

### Random Forest Model - Test Performance

In [ ]:
model_confusion_matrix(rf_model,X_test,y_test)  ## Complete the code to create confusion matrix for test data

In [ ]:
random_forest_perf_test = model_performance_classification(rf_model,X_test,y_test)  ## Complete the code to check performance on test data
random_forest_perf_test

# Model Performance Improvement

## Model Performance Improvement - Logistic Regression

In [ ]:
def calculate_vif(idf):
    """
    Calculate Variance Inflation Factor (VIF) for each variable in a DataFrame.

    Parameters:
    df (DataFrame): Input DataFrame containing numerical variables.

    Returns:
    vif_df (DataFrame): DataFrame containing variable names and their corresponding VIF values.
    """
    variables = idf.values
    vif_df = pd.DataFrame()
    vif_df["Variable"] = idf.columns
    vif_df["VIF"] = [variance_inflation_factor(variables, i) for i in range(idf.shape[1])]
    return vif_df

In [ ]:
# Call the function to calculate VIF
vif_result = calculate_vif(X_train)  ## Complete the code to calculate VIF for the scaled X_train data

print("Variance Inflation Factors:")
vif_result

In [ ]:
high_vif_columns = []
for i, row in vif_result.iterrows():
    if row['VIF'] >= 5:
        high_vif_columns.append(row['Variable'])
high_vif_columns

In [ ]:
# Dropping columns with VIF > 5
X_train_scaled.drop(columns = high_vif_columns, axis=1, inplace=True)
X_test_scaled.drop(columns = high_vif_columns, axis=1, inplace=True)

In [ ]:
X_train_scaled.shape

In [ ]:
X_test_scaled.shape

In [ ]:
X_train_new_with_intercept = SM.add_constant(X_train_scaled)
X_test_new_with_intercept = SM.add_constant(X_test_scaled)

In [ ]:
# Retraining Logistic Regression Model with new data
#LogisticReg = 
LogisticReg_improved = SM.Logit(y_train, X_train_new_with_intercept)  ## Complete the code to fir Logistic Regression Model on new train data with intercept

# Fit the model
LogisticReg_improved_result = LogisticReg_improved.fit()

print(LogisticReg_improved_result.summary())

In [ ]:
# Finding Optimal Threshold value
logit_y_pred = LogisticReg_improved_result.predict(X_train_new_with_intercept)
fpr, tpr, thresholds = roc_curve(y_train, logit_y_pred)
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold_logit = round(thresholds[optimal_idx], 3)
optimal_threshold_logit

In [ ]:
roc_auc = roc_auc_score(y_train, logit_y_pred)  ## Complete the code to get roc_auc score
# Plot ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

### Logistic Regression Performance - Training Set

In [ ]:
model_confusion_matrix(LogisticReg_improved_result, X_train_new_with_intercept, y_train, optimal_threshold_logit)

In [ ]:
logistic_regression_tuned_perf_train = model_performance_classification(
    LogisticReg_improved_result, X_train_new_with_intercept, y_train, optimal_threshold_logit
)
logistic_regression_tuned_perf_train

### Logistic Regression Performance - Test Set

In [ ]:
model_confusion_matrix(LogisticReg_improved_result,X_test_new_with_intercept,y_test)  ## Complete the code to create confusion matrix for test data

In [ ]:
logistic_regression_tuned_perf_test = model_performance_classification(LogisticReg_improved_result,X_test_new_with_intercept,y_test)  ## Complete the code to check performance on test data
logistic_regression_tuned_perf_test

## Model Performance Improvement - Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],  # Number of trees in the forest
    'max_depth': [5, 7, 9],    # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],    # Minimum number of samples required to split a node
    'min_samples_leaf': [5, 6, 7],  # Minimum number of samples required at each leaf node
}

rf_classifier = RandomForestClassifier(class_weight='balanced', random_state=42)

grid_search = GridSearchCV(
    estimator=rf_classifier,
    param_grid=param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_search.fit(X_train_new_with_intercept, y_train)

print("Best parameters:", grid_search.best_params_)

In [ ]:
# Access the best estimator directly if needed
best_rf_classifier = grid_search.best_estimator_

In [ ]:
params_used = best_rf_classifier.get_params()

# Print the parameters
print("Parameters used in the Random Forest Classifier:")
for param_name, param_value in params_used.items():
    print(f"{param_name}: {param_value}")

### Random Forest Performance - Training Set

In [ ]:
model_confusion_matrix(best_rf_classifier,X_train_new_with_intercept, y_train)  ## Complete the code to create confusion matrix for training data

In [ ]:
random_forest_tuned_perf_train = model_performance_classification(best_rf_classifier,X_train_new_with_intercept, y_train)  ## Complete the code to check performance on training data
random_forest_tuned_perf_train

### Random Forest Performance - Test Set

In [ ]:
model_confusion_matrix(best_rf_classifier,X_test_new_with_intercept, y_test)  ## Complete the code to create confusion matrix for test data

In [ ]:
random_forest_tuned_perf_test = model_performance_classification(best_rf_classifier,X_test_new_with_intercept, y_test)  ## Complete the code to check performance on test data
random_forest_tuned_perf_test

# Model Comparison and Final Model Selection

In [ ]:
# training performance comparison

models_train_comp_df = pd.concat(
    [
        logistic_regression_perf_train.T,
        logistic_regression_tuned_perf_train.T,
        random_forest_perf_train.T,
        random_forest_tuned_perf_train.T,
    ],
    axis=1,
)
models_train_comp_df.columns = [
    "Logistic Regression",
    "Tuned Logistic Regression",
    "Random Forest",
    "Tuned Random Forest",
]
print("Training performance comparison:")
models_train_comp_df

In [ ]:
# testing performance comparison

models_test_comp_df = pd.concat(
    [
        logistic_regression_perf_test.T,
        logistic_regression_tuned_perf_test.T,
        random_forest_perf_test.T,
        random_forest_tuned_perf_test.T,
    ],
    axis=1,
)
models_test_comp_df.columns = [
    "Logistic Regression",
    "Tuned Logistic Regression",
    "Random Forest",
    "Tuned Random Forest",
]
print("Testing performance comparison:")
models_test_comp_df

In [ ]:
feature_names = X_train.columns
importances = best_rf_classifier.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(20, 20))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

# Insights and Recommendations

## Insights

1. Cash Flow Strength is a Critical Indicator of Default Risk
Companies with low values in features like Cash_Flow_Per_Share and Cash_Flow_to_Liability are significantly more likely to default.


2. Tax Rate and Equity Efficiency Reflect Financial Stress
A high Tax_rate_A (Effective Tax Rate) and low Net_Worth_Turnover_Rate_times (Equity Turnover) are strong predictors of default.


3. Leverage and Solvency Ratios Matter
Ratios like Interest_bearing_debt_interest_rate and Total_debt_to_Total_net_worth are consistently higher in defaulting companies.


4. Growth Metrics Alone Are Not Sufficient
Features like Total_Asset_Growth_Rate or Operating_Profit_Growth_Rate were not among the top predictors.

## Recommendations

1. Prioritize Cash Flow Metrics in Credit Evaluation
Incorporate cash flow–based ratios as primary indicators in the Financial Health Assessment Tool.


2. Develop a Risk Scoring System Based on Top Predictors
Build a transparent scoring model that weights features like Tax_rate_A, Cash_Flow_Per_Share, and Net_Worth_Turnover_Rate_times.


3. Enable Dynamic Threshold Adjustment for Risk Appetite
Allow users (e.g., investors or analysts) to adjust the classification threshold to balance recall and precision based on their tolerance for false positives.

4. Monitor Leverage and Solvency Trends Over Time
Integrate trend analysis for key ratios like Debt-to-Equity and Interest Coverage to detect deteriorating financial health early.

5. Educate Clients on Interpreting Financial Red Flags
Provide guidance within the tool to help users understand why a company is flagged—e.g., “High tax burden and low equity efficiency suggest elevated risk.”

6. Retrain and Validate the Model Periodically
Financial patterns evolve with market conditions. Regularly update the model with new fiscal data to maintain predictive accuracy and relevance.
